# Unit 2, Lecture 3: API integration

Last lecture the tools were functions on your machine. Now they reach a real
service over the network. The core idea is the **wrapper**: a thin translating
layer between a messy external API and the clean tool the model sees.

The rule the whole lecture is built on: **never let the shape of an external API
leak into your agent.**

Everything here runs offline. The network is behind a seam we can fake, which is
also the point of the last section.

## Setup

In [ ]:
from cse476.api import (
    HttpClient, FakeTransport,
    weather_api_tool, weather_tool_schema,
)

# In production you would hand HttpClient a real HTTP transport.
# Here we hand it a fake, so every fault is reproducible offline.
transport = FakeTransport()
client = HttpClient(transport=transport, base_url="https://api.example.com", api_key="secret")
get_weather = weather_api_tool(client)

print(get_weather("Delhi"))

## 1. What the API actually returns, versus what the model sees

The raw payload is nested, abbreviated, in Kelvin, with weather as a numeric
code. The wrapper turns all of that into one clean sentence.

In [ ]:
from cse476.api import _raw_weather_payload

raw = _raw_weather_payload("Delhi")
print("RAW payload the API hands you:")
print(raw)
print()
print("What the model sees after the wrapper:")
print(get_weather("Delhi"))

Look at the gap. The raw side has `tmp_k` (Kelvin), `cond_cd` (a code), and
nested `loc`/`cur` objects. The model side has none of it: just a readable
sentence. If you passed the raw payload to the model, you would be asking it to
know that 311 Kelvin is hot and 721 means haze, and paying for every token of
that noise on every call.

## 2. The key travels in a header, not the URL

A URL ends up in logs, history, and error messages. A header does not. The
wrapper puts the key in an `Authorization` header, and you can prove it never
leaks into the requested URL.

In [ ]:
transport = FakeTransport()
client = HttpClient(transport=transport, base_url="https://api.example.com", api_key="secret")
weather_api_tool(client)("Mumbai")

print("URLs actually requested:")
for u in transport.calls:
    print(" ", u)
print()
print("secret in any URL?", any("secret" in u for u in transport.calls))

## 3. The three failures, each handled differently

401, 429, and 503 are three different problems: your key, your timing, their
server. Collapsing them into one 'error' throws away the information you need to
respond. Watch each produce a distinct, actionable message.

In [ ]:
for status in (401, 429, 503):
    t = FakeTransport(status=status)
    c = HttpClient(transport=t, base_url="https://x", api_key="k")
    print(f"{status} -> {weather_api_tool(c)('Delhi')}")

## 4. The silent failures: timeout, and a 200 that lies

A timeout must not hang forever, and a 200 with an unparseable body is the
network version of last lecture's 'it lies'. Both become readable sentences, not
crashes.

In [ ]:
# a timeout
t = FakeTransport(raise_timeout=True)
c = HttpClient(transport=t, base_url="https://x", api_key="k")
print("timeout   ->", weather_api_tool(c)("Delhi"))

# a 200 with a broken body
t = FakeTransport(bad_json=True)
c = HttpClient(transport=t, base_url="https://x", api_key="k")
print("bad body  ->", weather_api_tool(c)("Delhi"))

## 5. The seam: testing network code without a network

Every cell above hit a different network fault, and not one touched a real
network. The only thing that changed each time was **which transport we handed
in**. Everything above the seam, the wrapper, the tool, was byte-for-byte
identical. That is what makes a fake trustworthy: it proves something true about
production, not about the test.

In [ ]:
# the schema the model sees mentions none of the mess
import json as _j
print(_j.dumps(weather_tool_schema(), indent=2))

## Your turn

**1. Wrap a second endpoint.** Add a `get_forecast` tool that reuses the same
`HttpClient`. Notice how little new code it takes once the client exists, that is
the wrapper pattern paying off.

**2. Test a failure without a network.** Write a check using `FakeTransport` that
proves your tool says something sensible on a 503. You will not touch the real
network once.

**3. Spot the leak.** Think of an API whose raw response you once passed straight
into your code. What would have broken the day it renamed one field? The wrapper
is what would have saved you. Write down the one field that would have hurt.

In [ ]:
# your work here
